In [ ]:
---
layout: post 
title: Gamify Exploration 
description: A game design lesson on sprite sheets, chase logic, sprite swapping, and NPC interaction systems
date: 2026-04-12 00:00:00 +0000
permalink: /characters-lesson/
hide: true
toc: false
author: Lance Oberiano, Yiming Yin, Arjun Ganesh
---


In [ ]:
%%js

// GAME_RUNNER: CSSE Zone Game | hide_edit: true,  width: 100%, height: 500px

import GameControl from '{{site.baseurl}}/assets/js/GameEnginev1.1/essentials/GameControl.js';
import GameLevelAquaticGameLevel from '{{site.baseurl}}/assets/js/projects/characters/levels/GameLevelAquaticGameLevel.js';
import GameLevelSeek from '{{site.baseurl}}/assets/js/projects/characters/levels/GameLevelSeek.js';
import GameLevelBasketball from '{{site.baseurl}}/assets/js/projects/characters/levels/GameLevelBasketball.js';

const findHeadingByKeywords = (keywords) => {
  const headings = Array.from(document.querySelectorAll('h1, h2, h3, h4'));
  return headings.find((heading) => {
    const text = (heading.textContent || '').toLowerCase();
    return keywords.every((keyword) => text.includes(keyword));
  }) || null;
};

const scrollToConceptForLevel = (levelName) => {
  const level = String(levelName || '').toLowerCase();
  const keywordMap = {
    aquatic: [
      ['concept 1', 'interaction'],
      ['interaction']
    ],
    seek: [
      ['concept 2', 'sprite', 'swap'],
      ['sprite', 'swap']
    ],
    basketball: [
      ['concept 3', 'chase'],
      ['chase']
    ]
  };

  const keywordSets = keywordMap[level];
  if (!keywordSets) return;

  const targetHeading = keywordSets
    .map((keywords) => findHeadingByKeywords(keywords))
    .find(Boolean);
  if (!targetHeading) return;

  targetHeading.scrollIntoView({ behavior: 'smooth', block: 'start' });
};

if (!window.__charactersLessonAutoScrollInstalled) {
  const handleLessonScrollEvent = (event) => {
    const level = event?.detail?.level;
    setTimeout(() => scrollToConceptForLevel(level), 250);
  };

  window.addEventListener('characters:level-complete', handleLessonScrollEvent);
  window.addEventListener('characters:concept-focus', handleLessonScrollEvent);
  window.__charactersLessonAutoScrollInstalled = true;
}

export const gameLevelClasses = [GameLevelAquaticGameLevel, GameLevelSeek, GameLevelBasketball];
export { GameControl };

## Concept 1 — Message & interaction logic (`GameLevelAquaticGameLevel`)

Every interactable object — Mermaid, Slime, starfish, trash — exposes an `interact()` method on its data object. The engine calls it when the player is close and presses the action key. The method body is just a regular function, so it can branch on game state, open dialogues, spawn items, or advance quests.

```mermaid
flowchart TD
  A[Player presses interact near Mermaid] --> B{Quest 1 accepted?}
  B -- No --> C[Mermaid asks player to accept Aquatic Quest 1]
  C --> D[Player accepts quest]
  B -- Yes --> E[Mermaid provides Aquatic Quest 1 status]
  D --> F[Track starfish collected]
  E --> F
  F --> G{Starfish requirement met?}
  G -- No --> H[Mermaid says collect more starfishes]
  H --> F
  G -- Yes --> I[Complete Aquatic Quest 1]
  I --> J[Unlock Aquatic Quest 2 dialogue]
  J --> K[Slime NPC becomes next quest contact]
```

```js
interact: function() {
  if (!this.dialogueSystem) return;

  const q1 = questState.firstQuest;
  if (q1.accepted) {
    this.dialogueSystem.showDialogue(
      'Keep collecting starfishes!', 'Mermaid', null
    );
    return;
  }
  // First interaction: show accept/decline buttons
  this.dialogueSystem.showDialogue(
    "I've lost all my starfishes. Will you collect them?",
    'Mermaid', null
  );
  this.dialogueSystem.addButtons([
    { text: 'Accept', primary: true, action: () => { q1.accepted = true; } },
    { text: 'Decline', action: () => this.dialogueSystem.closeDialogue() }
  ]);
}
```

Dialogue branching is driven by a plain state machine object — no class hierarchy needed. Each flag is a boolean or number that the dialogue methods read before deciding which text to show:

```js
const questState = {
  firstQuest:  { accepted: false, completed: false, collected: 0, starfishTotal: 8 },
  secondQuest: { accepted: false, inSurface: false, completed: false, collected: 0 }
};
```

Collectibles like starfish use the same `interact()` hook but do something lighter — increment a counter, call `this.destroy()` to remove themselves from the scene, then update the HUD:

```js
interact: function() {
  questState.firstQuest.collected += 1;
  updateQuestHud();
  this.destroy();          // removes canvas element + splices from gameObjects
}
```

A recurring problem with multi-step dialogues is stale action buttons accumulating in the DOM. The level solves this with a `clearDialogueActionButtons()` helper that walks the dialogue box's child nodes and removes any flex containers that are not the avatar row:

```js
const clearDialogueActionButtons = (dialogueSystem) => {
  const buttons = dialogueSystem.dialogueBox
    .querySelectorAll('div[style*="display: flex"]');
  buttons.forEach((el) => {
    if (!el.contains(avatarElement)) el.remove();
  });
};
```



# Concept 2 — Character Swap Menu (Seek Level)

Press **Q** to open or close the character menu.
The menu is a simple popup (`div`) with buttons for each character.

When a button is clicked, the player's sprite changes **without creating a new player**.

**Key idea:**
We update the existing player instead of replacing it.

```js
// Press Q to toggle the menu
document.addEventListener("keydown", (e) => {
    if (e.key.toLowerCase() === "q") {
        e.preventDefault();
        toggleMenu();
    }
});
```

## Toggling the Menu

The menu is shown or hidden by checking its current state.

```js
const toggleMenu = () => {
    const isOpen = spriteMenu.style.display === 'block';
    setMenuVisibility(!isOpen);
};
```

## Prevent Duplicate Menus

Before creating a menu, the game removes any existing one.
This avoids multiple menus stacking on top of each other.

```js
const existingMenu = document.getElementById(menuId);
if (existingMenu) existingMenu.remove();
```

## Changing the Player Sprite

When a character is selected:

* The sprite image is updated
* The same player object is reused

The game waits for the image to load before rendering it.

```js
player.spriteReady = false;

player.spriteSheet.onload = () => {
    player.spriteReady = true;
    player.resize();
};

player.spriteSheet.src = spriteOption.src;
```

## Supporting Different Characters

Each character can have different:

* Sprite layouts (rows/columns)
* Sizes (scale)
* Animation speeds

One system handles all of them.

```js
const spriteOptions = [
    { label: "Boy",   orientation: { rows: 4, columns: 3  }, SCALE_FACTOR: 5,  ANIMATION_RATE: 50  },
    { label: "Kirby", orientation: { rows: 1, columns: 13 }, SCALE_FACTOR: 7,  ANIMATION_RATE: 8   },
    { label: "Astro", orientation: { rows: 4, columns: 4  }, SCALE_FACTOR: 11, ANIMATION_RATE: 110 },
];
```



# Concept 3 - Chase Logic

---

## Controls (Basketball Level Reference)

| Key | Action |
|---|---|
| WASD | Move the player |
| E | Shoot a basketball (stuns Kirby 3 s) |
| R | Restart the round after being caught |

---

## PART 1 — Finding the Player and Enemy

Before anything can happen, the game needs to find both the player and the chaser. This runs **every frame** inside `update()`.

```js
const player = this.gameEnv.gameObjects.find(obj => obj?.spriteData?.id === 'BasketballPlayer');
const lebron = this.gameEnv.gameObjects.find(obj => obj?.spriteData?.id === 'LeBron');
if (!player || !lebron) return;
```

- gameObjects = everything currently in the game
- .find() = searches for the object with that ID
- !player || !lebron = checks if one is missing
- return = stops the code early so the game doesn’t crash


## PART 2 - Moving Towards the Player
```js
const dx = player.position.x - lebron.position.x;
const dy = player.position.y - lebron.position.y;
lebron.position.x += dx;
lebron.position.y += dy;
```
What's Going On:

- dx = how far left/right the player is
- dy = how far up/down the player is
- This creates a direction toward the player
- The enemy moves directly using that direction
- Problem: movement is way too fast because it depends on distance

## PART 3 - Fixing Speed with Direction
```js
const dx = player.position.x - lebron.position.x;
const dy = player.position.y - lebron.position.y;
const dist = Math.hypot(dx, dy);

lebron.position.x += (dx / dist) * speed;
lebron.position.y += (dy / dist) * speed;
```

- What’s happening:

- dx and dy point toward the player
- dist is how far away the player is
- Dividing by dist removes the distance
- This leaves only direction
- Multiplying by speed sets how fast the enemy moves

- Key idea:

- Divide by distance → get direction
- Multiply by speed → control movement


## PART 4 - Unique Changes

### Speed Scaling
```js
const speed = Math.min(2.1 + this.currentTime * 0.03, 2.8);
```
- What's Going On:

- Starts at a base speed (2.1)
- Increases over time (currentTime)
- Math.min caps the speed so it doesn’t get too fast
- This makes the game gradually harder but still fair

### Face the Direction of Player
```
if (Math.abs(dx) > Math.abs(dy)) {
  lebron.direction = dx >= 0 ? 'right' : 'left';
} else {
  lebron.direction = dy >= 0 ? 'down' : 'up';
}
```
- What's Going On:

- Compares horizontal vs vertical movement
- Moves in the stronger direction
- Updates sprite direction (left/right/up/down)
- Makes movement look natural and responsive

### Collision Detection
```
if (this.isHitboxCollision(player, lebron)) {
  this.caught = true;
  this.showCaughtMessage();
}
```
- What's Going On:

- Checks if player and enemy overlap
- Uses hitboxes (invisible rectangles)
- If they touch → player is caught
- Triggers game over / reset behavior

## Final Idea

Press **Q** → choose a character → update sprite → keep playing instantly.

Everything works by updating the player’s properties in real time.

| Hook / method | When it fires | Typical body |
|---|---|---|
| `interact()` | Player presses action key while colliding | Open dialogue or collect item |
| `reaction()` | Passive collision without key press | Usually a no-op (`{}`) to suppress default popups |
| `destroy()` | Called by `interact()` on collectibles | Removes canvas, splices from `gameObjects` |
| `showDialogue(text, name, null)` | Inside `interact()` | Renders dialogue box with NPC name header |
| `addButtons([...])` | After `showDialogue()` | Appends action buttons with inline callbacks |

*All three concepts compose: interaction logic (Concept 1), live sprite swaps (Concept 2), and chase behavior (Concept 3). That composition is what powers the full multi-level flow.*


## Practice Hacks

Use the three mini-runners below to practice each logic in a much simpler form. Each runner already includes a visible proof state so you can test your changes quickly.

1. **Hack 1 — Chase logic:** the chaser should move toward the player every frame.
2. **Hack 2 — Swap logic:** press `E` near the swap coach to cycle characters, matching the `GameLevelSeek` interaction pattern.
3. **Hack 3 — Interaction logic:** press `E` to accept a quest, collect the item, then return to finish it.

**Tip:** if you want students to start from a harder version, remove one small block of logic from the runner and have them rebuild it.


In [ ]:
%%js

// GAME_RUNNER: Hack 1 - Chase Logic Drill | width: 100%, height: 420px

import GameControl from '{{site.baseurl}}/assets/js/GameEnginev1.1/essentials/GameControl.js';
import { GameEnvBackground, Player, NPC } from '{{site.baseurl}}/assets/js/GameEnginev1.1/essentials/Imports.js';

class ChaseHackLevel {
  constructor(gameEnv) {
    this.gameEnv = gameEnv;
    const path = gameEnv.path;
    const width = gameEnv.innerWidth;
    const height = gameEnv.innerHeight;
    this.startTime = 0;
    this.proofShown = false;
    this.enemyDistanceMoved = 0;
    this.lastEnemyPosition = null;
    this.hudId = 'hack-chase-hud';
    this.bannerId = 'hack-chase-proof';

    this.classes = [
      {
        class: GameEnvBackground,
        data: {
          name: 'hack_chase_bg',
          src: path + '/images/projects/characters/tagplayground.png',
          pixels: { height: 400, width: 560 }
        }
      },
      {
        class: Player,
        data: {
          id: 'HackPlayer',
          src: path + '/images/projects/characters/boysprite.png',
          SCALE_FACTOR: 5,
          STEP_FACTOR: 1000,
          ANIMATION_RATE: 50,
          INIT_POSITION: { x: 50, y: 280 },
          pixels: { height: 612, width: 408 },
          orientation: { rows: 4, columns: 3 },
          down: { row: 0, start: 0, columns: 3 },
          downRight: { row: 1, start: 0, columns: 3, rotate: Math.PI / 16 },
          downLeft: { row: 0, start: 0, columns: 3, rotate: -Math.PI / 16 },
          left: { row: 2, start: 0, columns: 3 },
          right: { row: 1, start: 0, columns: 3 },
          up: { row: 3, start: 0, columns: 3 },
          upLeft: { row: 2, start: 0, columns: 3, rotate: Math.PI / 16 },
          upRight: { row: 3, start: 0, columns: 3, rotate: -Math.PI / 16 },
          hitbox: { widthPercentage: 0.15, heightPercentage: 0.15 },
          keypress: { up: 87, left: 65, down: 83, right: 68 }
        }
      },
      {
        class: NPC,
        data: {
          id: 'HackChaser',
          greeting: false,
          src: path + '/images/projects/characters/kirby.png',
          SCALE_FACTOR: 7,
          ANIMATION_RATE: 8,
          INIT_POSITION: { x: width - 130, y: 70 },
          pixels: { height: 36, width: 569 },
          orientation: { rows: 1, columns: 13 },
          down: { row: 0, start: 0, columns: 13 },
          left: { row: 0, start: 0, columns: 13 },
          right: { row: 0, start: 0, columns: 13 },
          up: { row: 0, start: 0, columns: 13 },
          downRight: { row: 0, start: 0, columns: 13 },
          downLeft: { row: 0, start: 0, columns: 13 },
          upRight: { row: 0, start: 0, columns: 13 },
          upLeft: { row: 0, start: 0, columns: 13 },
          hitbox: { widthPercentage: 0.2, heightPercentage: 0.2 },
          reaction: function() {},
          interact: function() {}
        }
      }
    ];
  }

  initialize() {
    this.startTime = performance.now();
    this.createHud();
  }

  update() {
    const player = this.findById('HackPlayer');
    const chaser = this.findById('HackChaser');
    if (!player || !chaser) return;

    const dx = player.position.x - chaser.position.x;
    const dy = player.position.y - chaser.position.y;
    const dist = Math.hypot(dx, dy);

    if (dist > 1) {
      const speed = 1.9;
      chaser.position.x += (dx / dist) * speed;
      chaser.position.y += (dy / dist) * speed;
    }

    chaser.position.x = Math.max(0, Math.min(chaser.position.x, this.gameEnv.innerWidth - (chaser.width || 0)));
    chaser.position.y = Math.max(0, Math.min(chaser.position.y, this.gameEnv.innerHeight - (chaser.height || 0)));

    if (Math.abs(dx) > Math.abs(dy)) {
      chaser.direction = dx >= 0 ? 'right' : 'left';
    } else {
      chaser.direction = dy >= 0 ? 'down' : 'up';
    }

    if (this.lastEnemyPosition) {
      this.enemyDistanceMoved += Math.hypot(
        chaser.position.x - this.lastEnemyPosition.x,
        chaser.position.y - this.lastEnemyPosition.y
      );
    }
    this.lastEnemyPosition = { x: chaser.position.x, y: chaser.position.y };

    const elapsed = ((performance.now() - this.startTime) / 1000).toFixed(1);
    this.updateHud(dist, elapsed);

    if (!this.proofShown && this.enemyDistanceMoved > 150 && dist < 80) {
      this.showProof();
    }
  }

  findById(id) {
    return this.gameEnv.gameObjects.find(obj => obj?.spriteData?.id === id);
  }

  createHud() {
    this.removeDom();
    const hud = document.createElement('div');
    hud.id = this.hudId;
    hud.style.cssText = 'position:fixed;top:14px;left:14px;z-index:9999;background:rgba(7,16,32,0.88);color:#dbeafe;border:2px solid #60a5fa;border-radius:12px;padding:10px 14px;font:12px monospace;';
    hud.textContent = 'Use WASD. Watch the chaser follow the player.';
    document.body.appendChild(hud);
  }

  updateHud(dist, elapsed) {
    const hud = document.getElementById(this.hudId);
    if (!hud) return;
    hud.textContent = 'Survive: ' + elapsed + 's | Distance to chaser: ' + Math.round(dist) + ' | Goal: let the chaser close in.';
  }

  showProof() {
    this.proofShown = true;
    const banner = document.createElement('div');
    banner.id = this.bannerId;
    banner.style.cssText = 'position:fixed;right:14px;top:14px;z-index:10000;background:#14532d;color:#dcfce7;border:2px solid #4ade80;border-radius:12px;padding:12px 14px;font:12px monospace;max-width:320px;';
    banner.textContent = 'Proof unlocked: the chase loop is working. The enemy moved frame-by-frame toward the player.';
    document.body.appendChild(banner);
  }

  removeDom() {
    document.getElementById(this.hudId)?.remove();
    document.getElementById(this.bannerId)?.remove();
  }

  destroy() {
    this.removeDom();
  }
}

export const gameLevelClasses = [ChaseHackLevel];
export { GameControl };

In [ ]:
%%js

// GAME_RUNNER: Hack 2 - E Character Swap Drill | width: 100%, height: 420px

import GameControl from '{{site.baseurl}}/assets/js/GameEnginev1.1/essentials/GameControl.js';
import { GameEnvBackground, Player, NPC } from '{{site.baseurl}}/assets/js/GameEnginev1.1/essentials/Imports.js';

class SwapHackLevel {
  constructor(gameEnv) {
    this.gameEnv = gameEnv;
    const path = gameEnv.path;
    this.path = path;
    this.activeIndex = 0;
    this.swapCount = 0;
    this.proofShown = false;
    this.hudId = 'hack-swap-hud';
    this.bannerId = 'hack-swap-proof';
    this.playerSpawn = { x: 70, y: 265 };
    this.playerSkins = [
      {
        name: 'Boy Sprite',
        src: path + '/images/projects/characters/boysprite.png',
        pixels: { height: 612, width: 408 },
        scale: 5
      },
      {
        name: 'Scuba Diver',
        src: path + '/images/projects/characters/scubadiver.png',
        pixels: { height: 948, width: 632 },
        scale: 5
      }
    ];

    this.classes = [
      {
        class: GameEnvBackground,
        data: {
          name: 'hack_swap_bg',
          src: path + '/images/projects/characters/Aquatic.png',
          pixels: { height: 1960, width: 2940 }
        }
      },
      { class: Player, data: this.createPlayerData(this.activeIndex, this.playerSpawn) },
      {
        class: NPC,
        data: {
          id: 'SwapCoach',
          greeting: false,
          src: path + '/images/projects/characters/robot.png',
          SCALE_FACTOR: 7,
          ANIMATION_RATE: 50,
          INIT_POSITION: { x: 370, y: 180 },
          pixels: { height: 256, width: 256 },
          orientation: { rows: 1, columns: 1 },
          down: { row: 0, start: 0, columns: 1 },
          left: { row: 0, start: 0, columns: 1 },
          right: { row: 0, start: 0, columns: 1 },
          up: { row: 0, start: 0, columns: 1 },
          downRight: { row: 0, start: 0, columns: 1 },
          downLeft: { row: 0, start: 0, columns: 1 },
          upRight: { row: 0, start: 0, columns: 1 },
          upLeft: { row: 0, start: 0, columns: 1 },
          hitbox: { widthPercentage: 0.1, heightPercentage: 0.1 },
          reaction: function() {},
          interact: () => this.swapCharacter()
        }
      }
    ];
  }

  createPlayerData(index, position) {
    const skin = this.playerSkins[index];
    return {
      id: 'SwapPlayer',
      greeting: 'Current skin: ' + skin.name,
      src: skin.src,
      SCALE_FACTOR: skin.scale,
      STEP_FACTOR: 1000,
      ANIMATION_RATE: 50,
      INIT_POSITION: position,
      pixels: skin.pixels,
      orientation: { rows: 4, columns: 3 },
      down: { row: 0, start: 0, columns: 3 },
      downRight: { row: 1, start: 0, columns: 3, rotate: Math.PI / 16 },
      downLeft: { row: 0, start: 0, columns: 3, rotate: -Math.PI / 16 },
      left: { row: 2, start: 0, columns: 3 },
      right: { row: 1, start: 0, columns: 3 },
      up: { row: 3, start: 0, columns: 3 },
      upLeft: { row: 2, start: 0, columns: 3, rotate: Math.PI / 16 },
      upRight: { row: 3, start: 0, columns: 3, rotate: -Math.PI / 16 },
      hitbox: { widthPercentage: 0.15, heightPercentage: 0.15 },
      keypress: { up: 87, left: 65, down: 83, right: 68 }
    };
  }

  initialize() {
    this.createHud();
    this.updateHud('Walk into the coach and press E to swap characters.');
  }

  swapCharacter() {
    const player = this.findById('SwapPlayer');
    if (!player) return;

    const nextIndex = (this.activeIndex + 1) % this.playerSkins.length;
    const nextPosition = { x: player.position.x, y: player.position.y };
    player.destroy();

    const swappedPlayer = new Player(this.createPlayerData(nextIndex, nextPosition), this.gameEnv);
    this.gameEnv.gameObjects.push(swappedPlayer);

    this.activeIndex = nextIndex;
    this.swapCount += 1;
    this.updateHud('Swapped to ' + this.playerSkins[this.activeIndex].name + '. Swaps: ' + this.swapCount);

    if (!this.proofShown && this.swapCount >= 2) {
      this.showProof();
    }
  }

  findById(id) {
    return this.gameEnv.gameObjects.find(obj => obj?.spriteData?.id === id);
  }

  createHud() {
    this.removeDom();
    const hud = document.createElement('div');
    hud.id = this.hudId;
    hud.style.cssText = 'position:fixed;top:14px;left:14px;z-index:9999;background:rgba(30,41,59,0.9);color:#e2e8f0;border:2px solid #facc15;border-radius:12px;padding:10px 14px;font:12px monospace;max-width:330px;';
    document.body.appendChild(hud);
  }

  updateHud(message) {
    const hud = document.getElementById(this.hudId);
    if (!hud) return;
    hud.textContent = message;
  }

  showProof() {
    this.proofShown = true;
    const banner = document.createElement('div');
    banner.id = this.bannerId;
    banner.style.cssText = 'position:fixed;right:14px;top:14px;z-index:10000;background:#78350f;color:#fef3c7;border:2px solid #facc15;border-radius:12px;padding:12px 14px;font:12px monospace;max-width:320px;';
    banner.textContent = 'Proof unlocked: the E-to-swap interaction worked and the player character changed live in the level.';
    document.body.appendChild(banner);
  }

  removeDom() {
    document.getElementById(this.hudId)?.remove();
    document.getElementById(this.bannerId)?.remove();
  }

  destroy() {
    this.removeDom();
  }
}

export const gameLevelClasses = [SwapHackLevel];
export { GameControl };

In [ ]:
%%js

// GAME_RUNNER: Hack 3 - Interaction Logic Drill | width: 100%, height: 420px

import GameControl from '{{site.baseurl}}/assets/js/GameEnginev1.1/essentials/GameControl.js';
import { GameEnvBackground, Player, NPC } from '{{site.baseurl}}/assets/js/GameEnginev1.1/essentials/Imports.js';
import Collectible from '{{site.baseurl}}/assets/js/GameEnginev1.1/essentials/Collectible.js';

class InteractionHackLevel {
  constructor(gameEnv) {
    this.gameEnv = gameEnv;
    const path = gameEnv.path;
    this.path = path;
    this.questState = {
      accepted: false,
      collected: 0,
      goal: 1,
      completed: false,
      itemSpawned: false
    };
    this.hudId = 'hack-interact-hud';
    this.bannerId = 'hack-interact-proof';
    this.proofShown = false;

    this.classes = [
      {
        class: GameEnvBackground,
        data: {
          name: 'hack_interact_bg',
          src: path + '/images/projects/characters/Aquatic.png',
          pixels: { height: 1960, width: 2940 }
        }
      },
      {
        class: Player,
        data: {
          id: 'QuestPlayer',
          src: path + '/images/projects/characters/scubadiver.png',
          SCALE_FACTOR: 5,
          STEP_FACTOR: 1000,
          ANIMATION_RATE: 50,
          INIT_POSITION: { x: 70, y: 265 },
          pixels: { height: 948, width: 632 },
          orientation: { rows: 4, columns: 3 },
          down: { row: 0, start: 0, columns: 3 },
          downRight: { row: 1, start: 0, columns: 3, rotate: Math.PI / 16 },
          downLeft: { row: 0, start: 0, columns: 3, rotate: -Math.PI / 16 },
          left: { row: 2, start: 0, columns: 3 },
          right: { row: 1, start: 0, columns: 3 },
          up: { row: 3, start: 0, columns: 3 },
          upLeft: { row: 2, start: 0, columns: 3, rotate: Math.PI / 16 },
          upRight: { row: 3, start: 0, columns: 3, rotate: -Math.PI / 16 },
          hitbox: { widthPercentage: 0.15, heightPercentage: 0.15 },
          keypress: { up: 87, left: 65, down: 83, right: 68 }
        }
      },
      {
        class: NPC,
        data: {
          id: 'QuestGuide',
          greeting: 'Press E to talk.',
          src: path + '/images/projects/characters/robot.png',
          SCALE_FACTOR: 7,
          ANIMATION_RATE: 50,
          INIT_POSITION: { x: 360, y: 170 },
          pixels: { height: 256, width: 256 },
          orientation: { rows: 1, columns: 1 },
          down: { row: 0, start: 0, columns: 1 },
          left: { row: 0, start: 0, columns: 1 },
          right: { row: 0, start: 0, columns: 1 },
          up: { row: 0, start: 0, columns: 1 },
          downRight: { row: 0, start: 0, columns: 1 },
          downLeft: { row: 0, start: 0, columns: 1 },
          upRight: { row: 0, start: 0, columns: 1 },
          upLeft: { row: 0, start: 0, columns: 1 },
          hitbox: { widthPercentage: 0.1, heightPercentage: 0.1 },
          reaction: function() {},
          interact: () => this.handleGuideInteraction()
        }
      }
    ];
  }

  initialize() {
    this.createHud();
    this.updateHud('Walk to the guide and press E to start the quest.');
  }

  handleGuideInteraction() {
    const guide = this.findById('QuestGuide');
    if (!guide) return;

    if (!this.questState.accepted) {
      this.questState.accepted = true;
      this.spawnQuestItem();
      this.updateHud('Quest accepted. Collect the coin, then come back and press E again.');
      guide.dialogueSystem?.showDialogue('Quest accepted. Grab the coin and return to me.', 'Guide', null);
      return;
    }

    if (this.questState.accepted && this.questState.collected < this.questState.goal) {
      this.updateHud('Not done yet. Find the coin and return.');
      guide.dialogueSystem?.showDialogue('You still need the coin.', 'Guide', null);
      return;
    }

    if (!this.questState.completed) {
      this.questState.completed = true;
      this.updateHud('Quest complete. You used state-based interaction logic.');
      guide.dialogueSystem?.showDialogue('Quest complete. Nice branching logic.', 'Guide', null);
      this.showProof();
      return;
    }

    this.updateHud('Already complete. The guide remembers your state.');
  }

  spawnQuestItem() {
    if (this.questState.itemSpawned) return;
    this.questState.itemSpawned = true;

    const itemData = {
      id: 'QuestCoin',
      src: this.path + '/images/projects/characters/bitcoin.png',
      SCALE_FACTOR: 12,
      STEP_FACTOR: 0,
      ANIMATION_RATE: 1,
      INIT_POSITION: { x: 520, y: 250 },
      pixels: { height: 32, width: 32 },
      orientation: { rows: 1, columns: 1 },
      hitbox: { widthPercentage: 0.15, heightPercentage: 0.15 },
      greeting: 'Coin collected!',
      dialogues: ['Coin collected!'],
      reaction: function() {},
      showReactionDialogue: function() {},
      interact: () => {
        this.questState.collected += 1;
        this.updateHud('Coin collected. Return to the guide and press E.');
        const coin = this.findById('QuestCoin');
        coin?.destroy();
      }
    };

    const coin = new Collectible(itemData, this.gameEnv);
    coin.showReactionDialogue = function() {};
    this.gameEnv.gameObjects.push(coin);
  }

  findById(id) {
    return this.gameEnv.gameObjects.find(obj => obj?.spriteData?.id === id);
  }

  createHud() {
    this.removeDom();
    const hud = document.createElement('div');
    hud.id = this.hudId;
    hud.style.cssText = 'position:fixed;top:14px;left:14px;z-index:9999;background:rgba(15,23,42,0.92);color:#e0f2fe;border:2px solid #38bdf8;border-radius:12px;padding:10px 14px;font:12px monospace;max-width:360px;';
    document.body.appendChild(hud);
  }

  updateHud(message) {
    const hud = document.getElementById(this.hudId);
    if (!hud) return;
    hud.textContent = message + ' | Progress: ' + this.questState.collected + '/' + this.questState.goal;
  }

  showProof() {
    if (this.proofShown) return;
    this.proofShown = true;
    const banner = document.createElement('div');
    banner.id = this.bannerId;
    banner.style.cssText = 'position:fixed;right:14px;top:14px;z-index:10000;background:#164e63;color:#cffafe;border:2px solid #22d3ee;border-radius:12px;padding:12px 14px;font:12px monospace;max-width:340px;';
    banner.textContent = 'Proof unlocked: your interaction logic branched correctly from accept -> collect -> return -> complete.';
    document.body.appendChild(banner);
  }

  removeDom() {
    document.getElementById(this.hudId)?.remove();
    document.getElementById(this.bannerId)?.remove();
  }

  destroy() {
    this.removeDom();
  }
}

export const gameLevelClasses = [InteractionHackLevel];
export { GameControl };